# 📦 Bước 1: Load Dataset & Data Profiling (Chapter 2)
Dùng pandas để đọc file và chạy profiling cơ bản.
**Mục tiêu:** Phát hiện Null, Outlier, Sai định dạng.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Load data ---
df = pd.read_csv('dataset/listings.csv', low_memory=False)
print('Shape:', df.shape)
df.head(3)

In [ ]:
# --- Content Discovery: Null, Cardinality, Min/Max ---
def profiling_report(df):
    report = pd.DataFrame({
        'Dtype'       : df.dtypes,
        'Null_Count'  : df.isnull().sum(),
        'Completeness': (1 - df.isnull().mean()).map('{:.1%}'.format),
        'Cardinality' : df.nunique(),
        'Uniqueness'  : (df.nunique() / len(df)).map('{:.1%}'.format),
    })
    # Thêm min/max chỉ cho cột số
    num_cols = df.select_dtypes(include='number').columns
    report.loc[num_cols, 'Min'] = df[num_cols].min()
    report.loc[num_cols, 'Max'] = df[num_cols].max()
    return report

profiling_report(df)

# 🧹 Bước 2: Làm sạch theo Rule đã thống nhất (Chapter 2)
Rule chung cả nhóm:
- Xóa dòng `price` <= 0 hoặc null
- Chuẩn hóa cột ngày về `YYYY-MM-DD`
- Strip khoảng trắng, lower-case các cột categorical

In [ ]:
# --- Parse price (loại bỏ ký tự $, dấu phẩy) ---
if df['price'].dtype == object:
    df['price'] = df['price'].str.replace(r'[$,]', '', regex=True).astype(float)

# Rule 1: Xóa price <= 0
df = df[df['price'] > 0]

# Rule 2: Chuẩn hóa ngày
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

# Rule 3: Strip các cột text
str_cols = df.select_dtypes('object').columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

print(f'Sau clean: {df.shape}')
df.to_csv('dataset/listings_clean.csv', index=False)  # <<< Đây là Master CSV
print('✅ Đã lưu Master CSV!')

# 📐 Bước 3: Data Abstraction (Chapter 3)
Ghi chú lại kiểu dữ liệu và ngữ nghĩa của từng cột mình sẽ dùng.

In [ ]:
# Mỗi người điền vào đây các cột mình dùng
MY_ABSTRACTION = [
    # (Tên cột,   Loại C/O/Q,   Key/Value,   Direction,       Continuous/Discrete, Ý nghĩa)
    ('neighbourhood_group', 'C', 'V', '-',          'Discrete',   'Quận (Borough)'),
    ('room_type',           'C', 'V', '-',          'Discrete',   'Loại phòng'),
    ('price',               'Q', 'V', 'Sequential', 'Continuous', 'Giá/đêm (USD)'),
    ('number_of_reviews',   'Q', 'V', 'Sequential', 'Discrete',   'Số lượt review'),
    ('latitude',            ' ', 'V', '-',          'Continuous', 'Vĩ độ'),
    ('longitude',           ' ', 'V', '-',          'Continuous', 'Kinh độ'),
]

abs_df = pd.DataFrame(MY_ABSTRACTION,
    columns=['Column', 'Type(C/O/Q)', 'Key/Value', 'Direction', 'Cont/Disc', 'Semantic'])
abs_df

# 🎯 Bước 4: Task Abstraction (Chapter 4)
Chuyển câu hỏi kinh doanh thành `Action + Target` theo mô hình slide.

In [ ]:
# Mỗi người tự điền câu User Story + Action/Target của mình:
MY_TASKS = {
    'Task_X': {
        'User Story': 'As a [vai trò], I want to [mục tiêu], so that I can [hành động].',
        'Action'    : 'Consume > Discover',   # Discover / Present / Browse / Compare / Summarize
        'Target'    : 'All data > Trends',     # Trends / Outliers / Distribution / Dependency
    },
    'Task_Y': {
        'User Story': 'As a [vai trò], I want to [mục tiêu], so that I can [hành động].',
        'Action'    : 'Query > Compare',
        'Target'    : 'Attributes Many > Similarity',
    },
}

for task, info in MY_TASKS.items():
    print(f'--- {task} ---')
    for k, v in info.items():
        print(f'  {k}: {v}')

# 📊 Bước 5: Quick Plot (Kiểm tra nhanh trước khi sang D3)
Dùng matplotlib để kiểm tra phân phối dữ liệu trước khi code D3 chính thức.

In [ ]:
df = pd.read_csv('dataset/listings_clean.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: Phân phối price (cắt outlier cực đại) ---
price_99 = df['price'].quantile(0.99)
df[df['price'] <= price_99]['price'].hist(bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Phân phối Giá (bỏ top 1%)')
axes[0].set_xlabel('Price (USD)')

# --- Plot 2: Số listing theo Borough ---
df['neighbourhood_group'].value_counts().plot.bar(ax=axes[1], color='coral')
axes[1].set_title('Listing Count by Borough')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('quick_check.png', dpi=100)
plt.show()